# 8. Where does the encoder hesitate?

With `uncertainty=True`, every pixel's concept comes with an uncertainty: the normalised entropy of its
assignment, H(softmax(-d² / T)) / log K, where d² are the squared distances of the pixel's composite
feature to the K concept prototypes. It goes from 0 (one concept clearly wins) to 1 (all equally
likely); 1 - uncertainty is the certainty.

**What to look for:** uncertainty concentrated on boundaries means clear-cut concepts. Large uncertain
areas inside objects mean the encoder does not commit to one interpretation there.

In [ ]:
# %pip install nave-xai
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cbook
from matplotlib.image import imread
from nave_xai import NAVEImage

image = imread(cbook.get_sample_data("grace_hopper.jpg", asfileobj=False))   # ships with matplotlib

def show(ax, explainer, exp, title=""):
    """An explanation map in the explainer's concept colours."""
    ax.imshow(explainer.palette[exp.explanation]); ax.set_title(title, fontsize=10); ax.axis("off")

DINO = "dino"             # "random" to skip downloads
explainer = NAVEImage(model="vits16", training=DINO, k=8)
exp = explainer.explain(image, uncertainty=True)
colours = explainer.palette[exp.explanation] / 255
u = exp.uncertainty
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
show(axes[0], explainer, exp, "explanation")
axes[1].imshow(u, cmap="inferno", vmin=0, vmax=1); axes[1].set_title("uncertainty"); axes[1].axis("off")
axes[2].imshow((1 - u)[..., None] * colours + u[..., None])          # fades to white where uncertain
axes[2].set_title("opacity = certainty"); axes[2].axis("off");

### Is uncertainty a boundary effect?

Mean uncertainty as a function of the distance to the nearest region border. A steep decay means
uncertainty sits on the borders; a flat profile means it spreads inside regions too.

In [ ]:
distances = [1, 2, 4, 8, 16, 32]
profile = [u[exp.border(1) == 1].mean()]
for d in distances[1:]:
    ring = (exp.border(d) == 1) & (exp.border(distances[distances.index(d) - 1]) == 0)
    profile.append(u[ring].mean())
inside = u[exp.border(32) == 0].mean()
plt.plot(distances, profile, marker="o", label="within d px of a border")
plt.axhline(inside, ls="--", c="grey", label="further inside")
plt.xscale("log"); plt.xlabel("distance to the nearest border (px)"); plt.ylabel("mean uncertainty"); plt.legend();

### Temperature

`uncertainty_temp` sets how strictly concept scores are compared: lower values make the map more
contrasted, higher values softer. The default is 0.1; `"auto"` makes a typical pixel's two best
concepts differ by one unit.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4.5))
for ax, T in zip(axes, [0.03, 0.1, 1.0, "auto"]):
    e = NAVEImage(model="vits16", training=DINO, k=8, uncertainty_temp=T).explain(image, uncertainty=True)
    ax.imshow(e.uncertainty, cmap="inferno", vmin=0, vmax=1); ax.set_title(f"uncertainty_temp={T}"); ax.axis("off")

### Across a dataset

After `fit`, uncertainty maps share the same concepts, so they can be compared between images, for
instance to find the images an encoder finds hardest:

```python
explainer = NAVEImage(model="vits16", training="dino", k=8).fit(images)
hardest = np.argsort([e.uncertainty.mean() for e in explainer.predict(images, uncertainty=True)])[::-1]
```

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@article{boubekki2024nave,
  title   = {Multi-Depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  author  = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  journal = {arXiv preprint arXiv:2411.19700},
  year    = {2024}
}
```